# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gopinath04-R/gopinath-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Random Forest Classifier**

I'm choosing random forest because it already proved itself in the starter pipeline (Precision@50 of 0.740 vs the hand-written rule's 0.240) — it handles non-linear signal combinations that a manual rule can't, and it stays interpretable via feature importance, unlike a black-box neural net.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} pages. Method: RandomForestClassifier")

Loaded 30000 pages. Method: RandomForestClassifier


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split design: client-grouped holdout**

I'm splitting by client_id, not randomly, because pages from the same client can share patterns (site-wide templates, similar content). A random split would let the model "peek" at a client's other pages during training, inflating the score. This is the same client-holdout split the starter pipeline used, so it's a fair comparison against my Week-4 baseline.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

y = (df["trend_direction"] == "down").astype(int)
features = ["impressions_90d","sessions_90d","content_age_days","days_since_last_update",
            "avg_position","ctr","word_count","engagement_rate"]
X = df[features].replace([float("inf"), float("-inf")], None).fillna(0)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows | Client overlap: {len(set(df['client_id'].iloc[train_idx]) & set(df['client_id'].iloc[test_idx]))}")

Train: 23837 rows | Test: 6163 rows | Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training the random forest on the same split, same data, and comparing against my Week-4 baseline score using Precision@50 — the same metric.

In [3]:
def precision_at_k(scores, labels, k):
    order = pd.Series(scores).sort_values(ascending=False).index[:k]
    return labels.iloc[order].mean()

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

baseline_scores = ((df["days_since_last_update"].iloc[test_idx] >= 180) & (df["impressions_90d"].iloc[test_idx] >= 500)).astype(int) * 0.4 + \
                   ((df["trend_direction"].iloc[test_idx] == "down") & (df["impressions_90d"].iloc[test_idx] >= 100)).astype(int) * 0.35

for k in (20, 50):
    print(f"Baseline Precision@{k}: {precision_at_k(baseline_scores.reset_index(drop=True), y_test.reset_index(drop=True), k):.3f}")
    print(f"Random Forest Precision@{k}: {precision_at_k(pd.Series(rf_scores), y_test.reset_index(drop=True), k):.3f}")

Baseline Precision@20: 1.000
Random Forest Precision@20: 0.750
Baseline Precision@50: 1.000
Random Forest Precision@50: 0.720


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Looking at where the model is wrong, and what it leaned on most — a short error table beats a single accuracy number.

In [4]:
import numpy as np
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Top features:\n", importances.head(5))

pred = (rf_scores >= 0.5).astype(int)
errors = pd.DataFrame({"actual": y_test.values, "predicted": pred, "score": rf_scores})
false_pos = errors[(errors.actual == 0) & (errors.predicted == 1)]
false_neg = errors[(errors.actual == 1) & (errors.predicted == 0)]
print(f"False positives: {len(false_pos)} | False negatives: {len(false_neg)}")
print("Likely cause: false positives are pages with strong staleness/CTR signals that didn't actually decline (maybe protected by other factors); false negatives are declining pages that look 'healthy' on these features alone (e.g. decline driven by something outside this feature set).")

Top features:
 impressions_90d     0.222274
avg_position        0.202224
content_age_days    0.146199
word_count          0.130816
sessions_90d        0.117977
dtype: float64
False positives: 1451 | False negatives: 1283
Likely cause: false positives are pages with strong staleness/CTR signals that didn't actually decline (maybe protected by other factors); false negatives are declining pages that look 'healthy' on these features alone (e.g. decline driven by something outside this feature set).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.